In [ ]:
import ms.version
ms.version.addpkg('plotly', '5.8.2')
ms.version.addpkg('tenacity', '8.2.2')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from typing import Optional, List, Sequence


class InteractivePlotter:
    """
    Small helper class to create common interactive Plotly charts.

    NOTE:
    - Methods do NOT call fig.show().
    - They always return the "Figure".
    In jupyter, se lasci la chiamata come ultima riga della cella,
    il grafico viene mostrato una sola volta.
    """

    def __init__(self, df: Optional[pd.DataFrame] = None, template: str = "plotly_white") -> None:
        """
        Parameters
        ----------
        df : pd.DataFrame | None
            Optional default DataFrame to use for the methods.
        template : str
            Plotly template (e.g. 'plotly_white', 'plotly_dark').
        """
        self.df = df
        self.template = template

    # ---------------------------------------------------------------------
    # Utilities
    # ---------------------------------------------------------------------
    def set_data(self, df: pd.DataFrame) -> None:
        """Update the default DataFrame."""
        self.df = df

    # ---------------------------------------------------------------------
    # Time series
    # ---------------------------------------------------------------------
    def timeseries(
        self,
        col: str,
        df: Optional[pd.DataFrame] = None,
        title: Optional[str] = None,
        show_moving_average: bool = False,
        moving_average_window: int = 7,
    ) -> go.Figure:
        """
        Interactive time series from a single column.
        Assumes the index is a DatetimeIndex.
        """
        if df is None:
            df = self.df
        if df is None:
            raise ValueError("No DataFrame provided. Pass df=... or set self.df first.")

        if not isinstance(df.index, pd.DatetimeIndex):
            raise TypeError("The DataFrame index must be a DatetimeIndex.")

        if col not in df.columns:
            raise ValueError(f"Column '{col}' does not exist in the DataFrame.")

        series = df[col].sort_index()

        fig = go.Figure()

        # main series
        fig.add_trace(
            go.Scatter(
                x=series.index,
                y=series.values,
                mode="lines",
                name=col,
                hovertemplate="%{x|%Y-%m-%d %H:%M:%S}<br>" + col + ": %{y}<extra></extra>",
            )
        )

        # optional moving average
        if show_moving_average:
            ma = series.rolling(moving_average_window).mean()
            fig.add_trace(
                go.Scatter(
                    x=ma.index,
                    y=ma.values,
                    mode="lines",
                    name=f"MA ({moving_average_window})",
                    line=dict(dash="dash"),
                    hovertemplate="%{x|%Y-%m-%d %H:%M:%S}<br>MA: %{y}<extra></extra>",
                )
            )

        fig.update_layout(
            template=self.template,
            title=title or f"Time series - {col}",
            xaxis_title="Time",
            yaxis_title=col,
            hovermode="x unified",
            xaxis=dict(
                rangeselector=dict(
                    buttons=list(
                        [
                            dict(count=7, label="7d", step="day", stepmode="backward"),
                            dict(count=1, label="1m", step="month", stepmode="backward"),
                            dict(count=3, label="3m", step="month", stepmode="backward"),
                            dict(count=1, label="1y", step="year", stepmode="backward"),
                            dict(step="all", label="All"),
                        ]
                    ),
                ),
                rangeslider=dict(visible=True),
                type="date",
            ),
        )

        return fig

    # ---------------------------------------------------------------------
    # Bar from DataFrame (più barre affiancate per categoria)
    # ---------------------------------------------------------------------
    def bar(
        self,
        df: Optional[pd.DataFrame] = None,
        x: Optional[str] = None,
        y: Optional[Sequence[str]] = None,
        title: Optional[str] = None,
        mode: str = "group",  # "group" = side-by-side, "stack" = stacked
        sort_x: bool = False,
    ) -> go.Figure:
        """
        Interactive bar chart from a DataFrame, with multiple bars per category.
        """
        if df is None:
            df = self.df
        if df is None:
            raise ValueError("No DataFrame provided. Pass df=... or set self.df first.")

        data = df.copy()

        # se x è None, usa l'indice
        if x is None:
            data = data.reset_index()

        # *** converti tutti i nomi colonna in stringhe (Plotly-friendly) ***
        data.columns = [str(c) for c in data.columns]

        # ora x_col è semplicemente la prima colonna (ex index) oppure il nome passato
        x_col = data.columns[0] if x is None else str(x)

        # y: una o più colonne
        if y is None:
            y_cols = [c for c in data.columns if c != x_col]
        else:
            y_cols = [str(col) for col in y]

        if sort_x:
            data = data.sort_values(x_col)

        fig = px.bar(
            data,
            x=x_col,
            y=y_cols,
            barmode="group" if mode == "group" else "stack",
            title=title or "Bar chart",
            template=self.template,
        )

        fig.update_traces(hovertemplate="%{x}: %{y}<extra></extra>")
        fig.update_layout(
            xaxis_tickangle=-45,
            xaxis_title="",
            yaxis_title="",
        )

        return fig

    # ---------------------------------------------------------------------
    # Scatter
    # ---------------------------------------------------------------------
    def scatter(
        self,
        x: str,
        y: str,
        df: Optional[pd.DataFrame] = None,
        color: Optional[str] = None,
        size: Optional[str] = None,
        hover_data: Optional[List[str]] = None,
        title: Optional[str] = None,
    ) -> go.Figure:
        """
        Interactive scatter plot from a DataFrame.
        """
        if df is None:
            df = self.df
        if df is None:
            raise ValueError("No DataFrame provided. Pass df=... or set self.df first.")

        fig = px.scatter(
            df,
            x=x,
            y=y,
            color=color,
            size=size,
            hover_data=hover_data,
            title=title or f"{y} vs {x}",
            template=self.template,
        )

        fig.update_layout(
            xaxis_title="",
            yaxis_title="",
            hovermode="closest",
        )

        return fig
